In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.pipeline import FeatureUnion
import xgboost as xgb

In [ ]:
df_train = pd.read_csv("/kaggle/input/competitions/spaceship-titanic/train.csv")
df_test = pd.read_csv("/kaggle/input/competitions/spaceship-titanic/test.csv")
df_train.head()

# modify data and save to df_train_new

In [ ]:
df_train_new = df_train.copy()
df_test_new = df_test.copy()

# member_count 特徴量の生成
group = df_train["PassengerId"].str.split("_",expand=True).rename(columns={0:"group", 1:"id_in_group"})
member_count= group.groupby("group")["group"].transform("size")
df_train_new["member_count"] = member_count
group = df_test["PassengerId"].str.split("_",expand=True).rename(columns={0:"group", 1:"id_in_group"})
member_count= group.groupby("group")["group"].transform("size")
df_test_new["member_count"] = member_count

# deck, side 特徴量の生成
deck_side = df_train["Cabin"].str.split("/",expand=True).loc[:,[0,2]].rename(columns={0:"deck", 2:"side"})
df_train_new = pd.concat([df_train_new, deck_side], axis=1)
deck_side = df_test["Cabin"].str.split("/",expand=True).loc[:,[0,2]].rename(columns={0:"deck", 2:"side"})
df_test_new = pd.concat([df_test_new, deck_side], axis=1)

# Name Initial 特徴量の生成
df_train_new["First_init"] = df_train_new["Name"].str[0]
df_train_new["Last_init"] = df_train_new["Name"].str.extract(r"\b\s(\w)")
df_test_new["First_init"] = df_test_new["Name"].str[0]
df_test_new["Last_init"] = df_test_new["Name"].str.extract(r"\b\s(\w)")

# 不要な特徴量の削除
df_train_new = df_train_new.drop(columns=["Cabin","Name"])
df_test_new = df_test_new.drop(columns=["Cabin","Name"])

# VIPのdtypeをintに変換, 欠損値を0で埋める
df_train_new["VIP"] = df_train_new["VIP"].fillna(0).astype("int")
df_test_new["VIP"] = df_test_new["VIP"].fillna(0).astype("int")

# CryoSleepのNanはFalseで埋めておく
df_train_new["CryoSleep"] = df_train_new["CryoSleep"].fillna(False)
df_test_new["CryoSleep"] = df_test_new["CryoSleep"].fillna(False)

display(df_train_new.head())
display(df_test_new.head())

# notes
CryoSleepが単一の特徴量としては最も影響が強そう。\
CryoSleepは他の特徴量も大きく制限しているため、CryoSleepで場合分けしてモデリングする方が良いかもしれない

In [ ]:
df_train_new.info()

In [ ]:
df_train_new.nunique()

In [ ]:
df_train_new["Transported"].value_counts()

In [ ]:
df_train_new.groupby("CryoSleep")["Transported"].mean().plot(kind="bar")
df_train_new["CryoSleep"].value_counts()

In [ ]:
#sns.pairplot(df_train_new)

# CryoSleep = Trueの集団の特徴について

In [ ]:
#df_train_new.query("CryoSleep==True").plot(kind="bar", x="Age", y="")
# cryosleep = True の中で、transportedされた人の特徴を分析
agebin = pd.cut(df_train_new.query("CryoSleep==True")["Age"], bins=np.arange(0,85,5))
ax = df_train_new.query("CryoSleep==True").groupby(agebin, observed=False)["Transported"].mean().plot(kind="bar")
ax.set_title("Transported rate on age for CryoSleep=True")
plt.show()

ax = df_train_new.query("CryoSleep==True").groupby("member_count")["Transported"].mean().plot(kind="bar")
ax.set_title("Transported rate on member_count for CryoSleep=True")
plt.show()

t_rate_4Sleep = df_train_new.query("CryoSleep==True").groupby([agebin, "deck"],observed=False)["Transported"].mean()
ax = sns.heatmap(t_rate_4Sleep.unstack(), annot=True)
ax.set_title("Transported rate on each group of CryoSleep=True")
plt.show()

# membercountはあまり関係なさそう
#ABCDF deckはほとんどtransportedされている
# Europa Mars のHomePlanetはほとんどtranportedされている
# 55Cancrie のDestinationはほとんどtransportedされている
# sideはあまり関係なさそう

# CryoSleep = Falseの集団の特徴について

In [ ]:
agebin = pd.cut(df_train_new.query("CryoSleep==False")["Age"], bins=np.arange(0,85,5))
service_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
service = df_train_new.query("CryoSleep==False")[service_cols].sum(axis=1)
servicebin = pd.qcut(service, q=np.linspace(0,1,6), labels=np.linspace(0,1,6)[1:].round(1))

t_rate_4nonSleep = df_train_new.query("CryoSleep==False").groupby([agebin, "Last_init"],observed=False)["Transported"].mean()
ax = sns.heatmap(t_rate_4nonSleep.unstack(), annot=True)
ax.set_title("Transported rate on each group of CryoSleep=False")
plt.show()

ax = df_train_new.query("CryoSleep==False").groupby(servicebin,observed=False)["Transported"].mean().plot(kind="bar")
ax.set_title("Transported rate classified by service use sum")
ax.set_xlabel("service use quantile")
ax.set_ylabel("mean transported rate")
plt.show()


# ageは10歳以下がtransported高い
# deckはあまり関係なさそう
# member_countもあまり関係なさそう
# Europa のHomePlanet が若干transported高い
# 55 Cancri e のDestinationが若干高い
# S side が若干高い
# 名前はあまり関係なさそう
# トータルサービス利用が下位1割はtransportが高い

In [ ]:
from sklearn.feature_selection import mutual_info_regression
X = df_train_new.query("CryoSleep==False")[service_cols].fillna(0)
X["sum"] = X.sum(axis=1)
y = df_train_new.query("CryoSleep==False")["Transported"]
plt.bar(x=service_cols+["sum"], height=mutual_info_regression(X, y))
plt.suptitle("mutual info of kind of service to Transported")
plt.show()

# EDAまとめ

CryoSleepの有無で、データの性質が大きく異なる。これについて場合分けしてモデルを作るのが良さそう\
\
CryoSleep有の場合、変数はHomePlanet, Destination, Age, VIP, member_count, deck, side.\
このうち、deck, HomePlanet, Destination が特に重要。他の関係は薄そう\
\
CryoSleep無の場合、変数は上記に加え、RoomService, FoodCourt, ShoppingMall, Spa, VRDeckがある.\
これらのService利用料の合計が下位1割はTransported率が高い。\
他に、age, HomePlanet, Destination, sideが重要。 一方で、deckはあまり関係なさそう

In [ ]:
#binary_features = ["CryoSleep", "VIP"]
numeric_features = df_train_new.drop(columns="Transported").select_dtypes(exclude="object").columns.to_list()
df_train_nonnumeric = df_train_new.drop(columns=numeric_features+["Transported"])
categorical_features = df_train_nonnumeric.columns[df_train_nonnumeric.nunique()<=10].to_list()
remaining_features = df_train_nonnumeric.drop(columns=categorical_features).columns.to_list()

In [ ]:
print("numeric_features:", numeric_features)
print("categorical_features:", categorical_features)
print("remaining_features:", remaining_features)

# モデル実装

In [ ]:
# 実装
basic_features = ['HomePlanet','Destination','Age','VIP','member_count','deck','side']
service_features = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']

X_train_Cryo1 = df_train_new.query("CryoSleep==True")[basic_features]
y_train_Cryo1 = df_train_new.query("CryoSleep==True")["Transported"]
X_test_Cryo1 = df_test_new.query("CryoSleep==True")[basic_features]
id_test_Cryo1 = df_test_new.query("CryoSleep==True")["PassengerId"]

X_train_Cryo0 = df_train_new.query("CryoSleep==False")[basic_features+service_features]
y_train_Cryo0 = df_train_new.query("CryoSleep==False")["Transported"]
X_test_Cryo0 = df_test_new.query("CryoSleep==False")[basic_features+service_features]
id_test_Cryo0 = df_test_new.query("CryoSleep==False")["PassengerId"]


# 欠損値の穴埋め方法
# HomePlanet→other, Destination→other, Age→most_frequent, VIP→0, Service→0, deck→other, side→other
#impute_w_other_features = ["HomePlanet","Destination","deck","side"]

# pipe1
num_features_Cryo1 = X_train_Cryo1.select_dtypes(exclude="object").columns.to_list()
cat_features_Cryo1 = X_train_Cryo1.select_dtypes("object").columns.to_list()

num_pipe1 = SimpleImputer(strategy="most_frequent")

cat_pipe1 = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="other")),
    ("encode", OneHotEncoder(handle_unknown="ignore"))
]) 

preprocessor1 = ColumnTransformer([
    ("num", num_pipe1, num_features_Cryo1),
    ("cat", cat_pipe1, cat_features_Cryo1)
])

pipe1 = Pipeline([
    ("preprocess", preprocessor1),
    #("model", RandomForestClassifier(n_estimators=1000, random_state=0))
    ("model", xgb.XGBClassifier(n_estimators=1000,
                               learning_rate=0.1,
                               max_depth=5,
                               random_state=0))
])

pipe1.fit(X_train_Cryo1, y_train_Cryo1)



# pipe0
num_features_Cryo0 = X_train_Cryo0.select_dtypes(exclude="object").columns.to_list()
cat_features_Cryo0 = X_train_Cryo0.select_dtypes("object").columns.to_list()

num_impute_pipe0 = ColumnTransformer([
    ("age", SimpleImputer(strategy="most_frequent"), ["Age"]),
    ("service", SimpleImputer(strategy="constant", fill_value=0), service_features)
    ],
    verbose_feature_names_out=False
)
num_impute_pipe0.set_output(transform="pandas")

transformer = FunctionTransformer(
    lambda X: X.sum(axis=1).to_frame()
)
num_add_pipe0 = ColumnTransformer([
    ("keep", "passthrough",slice(None)),
    ("service_sum", transformer, service_features)
])

num_pipe0 = Pipeline([
    ("impute", num_impute_pipe0),
    ("add", num_add_pipe0)
])

cat_pipe0 = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="other")),
    ("encode", OneHotEncoder(handle_unknown="ignore"))
]) 

preprocessor0 = ColumnTransformer([
    ("num", num_pipe0, num_features_Cryo0),
    ("cat", cat_pipe0, cat_features_Cryo0)
])

pipe0 = Pipeline([
    ("preprocess", preprocessor0),
    #("model", RandomForestClassifier(n_estimators=1000, random_state=0))
    ("model", xgb.XGBClassifier(n_estimators=1000,
                               learning_rate=0.1,
                               max_depth=5,
                               random_state=0))
])

pipe0.fit(X_train_Cryo0, y_train_Cryo0)

display(pipe1)
display(pipe0)

In [ ]:
y_pred_Cryo1 = pd.DataFrame({
    "PassengerId":id_test_Cryo1, "Transported":pipe1.predict(X_test_Cryo1)
})
y_pred_Cryo0 = pd.DataFrame({
    "PassengerId":id_test_Cryo0, "Transported":pipe0.predict(X_test_Cryo0)
})

y_pred = pd.concat([y_pred_Cryo1, y_pred_Cryo0],axis=0).sort_index()
y_pred["Transported"] = y_pred["Transported"].astype("bool")
y_pred.to_csv("submission.csv", index=False)